# Step 1: Data Preparation and Mutually Exclusive Splits

## Dataset
- Dataset: COCO2014
- Set COCO_ROOT to the dataset root; the default is coco2014/
- Expected directories: annotations/, train2014/, and val2014/


## 1.1 Import Required Libraries


In [ ]:
import os
import json
import numpy as np
from collections import defaultdict, Counter
from pathlib import Path
import shutil
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported successfully!")

## 1.2 Define Class Splits


In [ ]:
BASE_LABELS_COCO = [
    'hair drier', 'clock', 'wine glass', 'book', 'cake', 'tie', 'motorcycle', 'sheep',
    'bottle', 'giraffe', 'cell phone', 'suitcase', 'remote', 'bench', 'mouse', 'carrot',
    'banana', 'train', 'sports ball', 'toothbrush', 'fire hydrant', 'airplane', 'tv',
    'bus', 'refrigerator', 'couch', 'knife', 'toilet', 'elephant', 'truck', 'parking meter',
    'car', 'potted plant', 'kite', 'skateboard', 'orange', 'horse', 'cat', 'tennis racket',
    'bowl', 'scissors', 'baseball glove', 'apple', 'traffic light', 'handbag', 'donut',
    'dog', 'hot dog', 'oven', 'umbrella', 'sink', 'pizza'
]

VAL_LABELS_COCO = [
    'cow', 'dining table', 'zebra', 'sandwich', 'bear', 'toaster', 'person', 'laptop',
    'bed', 'teddy bear', 'baseball bat', 'skis'
]

NOVEL_LABELS_COCO = [
    'bicycle', 'boat', 'stop sign', 'bird', 'backpack', 'frisbee', 'snowboard', 'surfboard',
    'cup', 'fork', 'spoon', 'broccoli', 'chair', 'keyboard', 'microwave', 'vase'
]

print(f"Base classes: {len(BASE_LABELS_COCO)}")
print(f"Validation classes: {len(VAL_LABELS_COCO)}")
print(f"Novel classes: {len(NOVEL_LABELS_COCO)}")
print(f"Total: {len(BASE_LABELS_COCO) + len(VAL_LABELS_COCO) + len(NOVEL_LABELS_COCO)}")

all_labels = BASE_LABELS_COCO + VAL_LABELS_COCO + NOVEL_LABELS_COCO
if len(all_labels) != len(set(all_labels)):
    print("WARNING: Duplicate categories found!")
else:
    print("No duplicate categories - all sets are mutually exclusive!")

## 1.3 Set Paths


In [ ]:
DATASET_ROOT = os.environ.get("COCO_ROOT", "coco2014")
ANNOTATIONS_DIR = os.path.join(DATASET_ROOT, "annotations")
TRAIN2014_DIR = os.path.join(DATASET_ROOT, "train2014")
VAL2014_DIR = os.path.join(DATASET_ROOT, "val2014")

OUTPUT_DIR = os.environ.get("DEPF_DATA_DIR", "coco2014_data")

print(f"Dataset root: {DATASET_ROOT}")
print(f"Annotations: {ANNOTATIONS_DIR}")
print(f"Train images: {TRAIN2014_DIR}")
print(f"Val images: {VAL2014_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

## 1.4 Map COCO Class Names to IDs

Load COCO class metadata and build a mapping from class names to category IDs.


In [ ]:
def load_coco_categories(annotation_file):
    """
Load COCO category metadata and return name-to-ID and ID-to-name mappings.
    """
    with open(annotation_file, 'r') as f:
        coco_data = json.load(f)
    
    cat_name_to_id = {}
    cat_id_to_name = {}
    for cat in coco_data['categories']:
        cat_name_to_id[cat['name']] = cat['id']
        cat_id_to_name[cat['id']] = cat['name']
    
    return cat_name_to_id, cat_id_to_name, coco_data

train_ann_file = os.path.join(ANNOTATIONS_DIR, "instances_train2014.json")
cat_name_to_id, cat_id_to_name, _ = load_coco_categories(train_ann_file)

print(f"Total COCO categories: {len(cat_name_to_id)}")
print("\nCategory mapping sample:")
for i, (name, id) in enumerate(list(cat_name_to_id.items())[:5]):
    print(f"  {name}: {id}")

## 1.5 Validate Classes Against the Official COCO Class List


In [ ]:
def validate_categories(label_list, split_name, cat_name_to_id):
    """Check which requested class names are present in the COCO category mapping."""
    missing = []
    found = []
    
    for label in label_list:
        if label in cat_name_to_id:
            found.append(label)
        else:
            missing.append(label)
    
    print(f"\n{split_name}:")
    print(f"  Found in COCO: {len(found)}/{len(label_list)}")
    
    if missing:
        print(f"  WARNING: {len(missing)} categories NOT found in COCO:")
        for m in missing:
            print(f"    - {m}")
    else:
        print(f"  All categories found!")
    
    return found, missing

base_found, base_missing = validate_categories(BASE_LABELS_COCO, "BASE_LABELS_COCO", cat_name_to_id)
val_found, val_missing = validate_categories(VAL_LABELS_COCO, "VAL_LABELS_COCO", cat_name_to_id)
novel_found, novel_missing = validate_categories(NOVEL_LABELS_COCO, "NOVEL_LABELS_COCO", cat_name_to_id)

## 1.6 Map Label Sets to Category IDs


In [ ]:
def build_label_sets(cat_name_to_id):
    """Convert the base, validation, and novel class names to COCO category IDs."""
    base_cat_ids = set([cat_name_to_id[name] for name in BASE_LABELS_COCO if name in cat_name_to_id])
    val_cat_ids = set([cat_name_to_id[name] for name in VAL_LABELS_COCO if name in cat_name_to_id])
    novel_cat_ids = set([cat_name_to_id[name] for name in NOVEL_LABELS_COCO if name in cat_name_to_id])
    
    print(f"\nLabel ID sets:")
    print(f"  Base: {len(base_cat_ids)} categories")
    print(f"  Validation: {len(val_cat_ids)} categories")
    print(f"  Novel: {len(novel_cat_ids)} categories")
    
    base_val_overlap = base_cat_ids & val_cat_ids
    base_novel_overlap = base_cat_ids & novel_cat_ids
    val_novel_overlap = val_cat_ids & novel_cat_ids
    
    if base_val_overlap or base_novel_overlap or val_novel_overlap:
        print("\nWARNING: Found overlapping categories between sets!")
        if base_val_overlap:
            print(f"  Base-Val overlap: {base_val_overlap}")
        if base_novel_overlap:
            print(f"  Base-Novel overlap: {base_novel_overlap}")
        if val_novel_overlap:
            print(f"  Val-Novel overlap: {val_novel_overlap}")
    else:
        print("\nNo overlap between sets - perfectly mutually exclusive!")
    
    return base_cat_ids, val_cat_ids, novel_cat_ids

base_cat_ids, val_cat_ids, novel_cat_ids = build_label_sets(cat_name_to_id)

## 1.7 Load COCO Annotations and Filter Samples

Keep samples whose labels belong entirely to one class set. Discard samples with labels spanning multiple sets. Read annotations from COCO2014 train2014 and val2014.


In [ ]:
def load_and_filter_annotations(ann_file, base_cat_ids, val_cat_ids, novel_cat_ids):
    """Load COCO annotations and assign samples to mutually exclusive class splits."""
    with open(ann_file, 'r') as f:
        coco_data = json.load(f)
    
    image_id_to_filename = {}
    for img in coco_data['images']:
        image_id_to_filename[img['id']] = img['file_name']
    
    image_id_to_anns = defaultdict(list)
    for ann in coco_data['annotations']:
        image_id_to_anns[ann['image_id']].append(ann)
    
    base_samples = []
    val_samples = []
    novel_samples = []
    mixed_samples = []
    
    for img_id, anns in image_id_to_anns.items():
        cat_ids = set([ann['category_id'] for ann in anns])
        
        is_base = cat_ids.issubset(base_cat_ids) and len(cat_ids) > 0
        is_val = cat_ids.issubset(val_cat_ids) and len(cat_ids) > 0
        is_novel = cat_ids.issubset(novel_cat_ids) and len(cat_ids) > 0
        
        set_count = sum([is_base, is_val, is_novel])
        
        if set_count == 0:
            continue
        elif set_count > 1:
            mixed_samples.append({
                'image_id': img_id,
                'filename': image_id_to_filename[img_id],
                'category_ids': list(cat_ids)
            })
        else:
            sample = {
                'image_id': img_id,
                'filename': image_id_to_filename[img_id],
                'category_ids': list(cat_ids)
            }
            
            if is_base:
                base_samples.append(sample)
            elif is_val:
                val_samples.append(sample)
            elif is_novel:
                novel_samples.append(sample)
    
    return base_samples, val_samples, novel_samples, mixed_samples

print("Loading and filtering train2014...")
train_ann_file = os.path.join(ANNOTATIONS_DIR, "instances_train2014.json")
train_base, train_val, train_novel, train_mixed = load_and_filter_annotations(
    train_ann_file, base_cat_ids, val_cat_ids, novel_cat_ids
)

print(f"\nTrain2014 filtering results:")
print(f"  Base samples: {len(train_base)}")
print(f"  Val samples: {len(train_val)}")
print(f"  Novel samples: {len(train_novel)}")
print(f"  Mixed samples (discarded): {len(train_mixed)}")

In [ ]:
print("Loading and filtering val2014...")
val_ann_file = os.path.join(ANNOTATIONS_DIR, "instances_val2014.json")
val_base, val_val, val_novel, val_mixed = load_and_filter_annotations(
    val_ann_file, base_cat_ids, val_cat_ids, novel_cat_ids
)

print(f"\nVal2014 filtering results:")
print(f"  Base samples: {len(val_base)}")
print(f"  Val samples: {len(val_val)}")
print(f"  Novel samples: {len(val_novel)}")
print(f"  Mixed samples (discarded): {len(val_mixed)}")

## 1.8 Combine Samples from COCO2014 train2014 and val2014


In [ ]:
final_base_samples = train_base + val_base
final_val_samples = train_val + val_val
final_novel_samples = train_novel + val_novel
total_mixed = len(train_mixed) + len(val_mixed)

print("="*60)
print("FINAL DATASET SPLIT RESULTS")
print("="*60)
print(f"\nBase set (52 classes):")
print(f"  Total samples: {len(final_base_samples)}")
print(f"  From train2014: {len(train_base)}")
print(f"  From val2014: {len(val_base)}")

print(f"\nValidation set (12 classes):")
print(f"  Total samples: {len(final_val_samples)}")
print(f"  From train2014: {len(train_val)}")
print(f"  From val2014: {len(val_val)}")

print(f"\nNovel set (16 classes):")
print(f"  Total samples: {len(final_novel_samples)}")
print(f"  From train2014: {len(train_novel)}")
print(f"  From val2014: {len(val_novel)}")

print(f"\nMixed samples (discarded): {total_mixed}")
print(f"  From train2014: {len(train_mixed)}")
print(f"  From val2014: {len(val_mixed)}")

print(f"\nTotal valid samples: {len(final_base_samples) + len(final_val_samples) + len(final_novel_samples)}")

## 1.9 Analyze Class Distributions


In [ ]:
def analyze_category_distribution(samples, cat_id_to_name, split_name):
    """Count samples for each category in a dataset split."""
    cat_counts = Counter()
    
    for sample in samples:
        for cat_id in sample['category_ids']:
            cat_counts[cat_id] += 1
    
    sorted_cats = cat_counts.most_common()
    
    print(f"\n{split_name} - Category distribution:")
    print(f"{'Category':<20} {'Count':>10}")
    print("-" * 35)
    
    for cat_id, count in sorted_cats:
        cat_name = cat_id_to_name.get(cat_id, f"Unknown_{cat_id}")
        print(f"{cat_name:<20} {count:>10}")
    
    print(f"\nTotal images: {len(samples)}")
    print(f"Total category instances: {sum(cat_counts.values())}")
    print(f"Average instances per image: {sum(cat_counts.values()) / len(samples):.2f}")
    
    return cat_counts

base_cat_counts = analyze_category_distribution(final_base_samples, cat_id_to_name, "Base Set")
val_cat_counts = analyze_category_distribution(final_val_samples, cat_id_to_name, "Validation Set")
novel_cat_counts = analyze_category_distribution(final_novel_samples, cat_id_to_name, "Novel Set")

## 1.10 Visualize Class Distributions


In [ ]:
def plot_category_distribution(cat_counts, cat_id_to_name, split_name, color='blue'):
    """Plot the per-class sample counts for a dataset split."""
    sorted_items = sorted(cat_counts.items(), key=lambda item: cat_id_to_name[item[0]])
    cat_names = [cat_id_to_name.get(cat_id, f"Unknown_{cat_id}") for cat_id, _ in sorted_items]
    counts = [count for _, count in sorted_items]
    
    plt.figure(figsize=(14, 6))
    bars = plt.bar(range(len(cat_names)), counts, color=color, alpha=0.7)
    plt.xticks(range(len(cat_names)), cat_names, rotation=45, ha='right')
    plt.xlabel('Category')
    plt.ylabel('Number of Samples')
    plt.title(f'{split_name} - Category Distribution')
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width() / 2, height, f'{int(height)}', ha='center', va='bottom', fontsize=8)
    plt.tight_layout()
    plt.show()


## 1.11 Analyze Multi-Label Distributions


In [ ]:
def analyze_multilabel_distribution(samples, cat_id_to_name, split_name):
    """
Report sample and label counts for a dataset split.
    """
    label_counts = [len(sample['category_ids']) for sample in samples]
    
    count_freq = Counter(label_counts)
    
    print(f"\n{split_name} - Multi-label distribution:")
    print(f"{'Labels per Image':<20} {'Count':>10} {'Percentage':>12}")
    print("-" * 47)
    
    for num_labels in sorted(count_freq.keys()):
        count = count_freq[num_labels]
        percentage = count / len(samples) * 100
        print(f"{num_labels:<20} {count:>10} {percentage:>11.2f}%")
    
    print(f"\nStatistics:")
    print(f"  Average labels per image: {np.mean(label_counts):.2f}")
    print(f"  Median labels: {np.median(label_counts):.1f}")
    print(f"  Max labels: {max(label_counts)}")
    
    plt.figure(figsize=(10, 5))
    plt.hist(label_counts, bins=range(1, max(label_counts)+2), 
             edgecolor='black', alpha=0.7)
    plt.xlabel('Number of Labels per Image')
    plt.ylabel('Number of Images')
    plt.title(f'{split_name} - Multi-label Distribution')
    plt.xticks(range(1, max(label_counts)+2))
    plt.grid(axis='y', alpha=0.3)
    plt.show()

analyze_multilabel_distribution(final_base_samples, cat_id_to_name, "Base Set")
analyze_multilabel_distribution(final_val_samples, cat_id_to_name, "Validation Set")
analyze_multilabel_distribution(final_novel_samples, cat_id_to_name, "Novel Set")

## 1.12 Save Dataset Splits


In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

dataset_split = {
    'metadata': {
        'description': 'COCO dataset with mutually exclusive label splits',
        'base_classes': BASE_LABELS_COCO,
        'val_classes': VAL_LABELS_COCO,
        'novel_classes': NOVEL_LABELS_COCO,
        'total_base_samples': len(final_base_samples),
        'total_val_samples': len(final_val_samples),
        'total_novel_samples': len(final_novel_samples),
        'total_mixed_discarded': total_mixed
    },
    'base_samples': final_base_samples,
    'val_samples': final_val_samples,
    'novel_samples': final_novel_samples
}

output_file = os.path.join(OUTPUT_DIR, 'coco_mutually_exclusive_split.json')
with open(output_file, 'w') as f:
    json.dump(dataset_split, f, indent=2)

print(f"Dataset split saved to: {output_file}")
print(f"File size: {os.path.getsize(output_file) / 1024 / 1024:.2f} MB")

## 1.13 Save the Category ID Mapping


In [ ]:
category_mapping = {
    'cat_id_to_name': cat_id_to_name,
    'cat_name_to_id': cat_name_to_id,
    'base_cat_ids': list(base_cat_ids),
    'val_cat_ids': list(val_cat_ids),
    'novel_cat_ids': list(novel_cat_ids),
    'base_label_names': BASE_LABELS_COCO,
    'val_label_names': VAL_LABELS_COCO,
    'novel_label_names': NOVEL_LABELS_COCO
}

mapping_file = os.path.join(OUTPUT_DIR, 'category_mapping.json')
with open(mapping_file, 'w') as f:
    json.dump(category_mapping, f, indent=2)

print(f"Category mapping saved to: {mapping_file}")

## 1.14 Generate a Summary Statistics Report


In [ ]:
report = f"""
{'='*70}
COCO MUTUALLY EXCLUSIVE DATASET SPLIT - SUMMARY REPORT
{'='*70}

1. CATEGORY DISTRIBUTION
{'-'*70}
   Base Set:        {len(BASE_LABELS_COCO):3d} classes
   Validation Set:  {len(VAL_LABELS_COCO):3d} classes
   Novel Set:       {len(NOVEL_LABELS_COCO):3d} classes
   TOTAL:           {len(BASE_LABELS_COCO) + len(VAL_LABELS_COCO) + len(NOVEL_LABELS_COCO):3d} classes

2. SAMPLE DISTRIBUTION
{'-'*70}
   Base Set:        {len(final_base_samples):6d} images
   Validation Set:  {len(final_val_samples):6d} images
   Novel Set:       {len(final_novel_samples):6d} images
   Mixed (discarded): {total_mixed:6d} images
   TOTAL VALID:     {len(final_base_samples) + len(final_val_samples) + len(final_novel_samples):6d} images

3. SOURCE DISTRIBUTION
{'-'*70}
   Base Set:
      From train2014: {len(train_base):6d} images
      From val2014:   {len(val_base):6d} images

   Validation Set:
      From train2014: {len(train_val):6d} images
      From val2014:   {len(val_val):6d} images

   Novel Set:
      From train2014: {len(train_novel):6d} images
      From val2014:   {len(val_novel):6d} images

4. MULTI-LABEL STATISTICS
{'-'*70}
   Base Set:
      Avg labels/image: {np.mean([len(s['category_ids']) for s in final_base_samples]):.2f}
      Max labels:       {max([len(s['category_ids']) for s in final_base_samples])}

   Validation Set:
      Avg labels/image: {np.mean([len(s['category_ids']) for s in final_val_samples]):.2f}
      Max labels:       {max([len(s['category_ids']) for s in final_val_samples])}

   Novel Set:
      Avg labels/image: {np.mean([len(s['category_ids']) for s in final_novel_samples]):.2f}
      Max labels:       {max([len(s['category_ids']) for s in final_novel_samples])}

5. OUTPUT FILES
{'-'*70}
   Dataset split:     {output_file}
   Category mapping:  {mapping_file}

{'='*70}
"""

print(report)

report_file = os.path.join(OUTPUT_DIR, 'split_report.txt')
with open(report_file, 'w') as f:
    f.write(report)

print(f"\nReport saved to: {report_file}")

## 1.15 Validate the Split with Sample Records


In [ ]:
def verify_samples(samples, cat_id_to_name, split_name, num_samples=5):
    """Print representative records from a dataset split."""
    print(f"\n{split_name} - Sample verification (first {num_samples} samples):")
    print("-" * 70)
    
    for i, sample in enumerate(samples[:num_samples]):
        cat_names = [cat_id_to_name.get(cid, f"Unknown_{cid}") 
                     for cid in sample['category_ids']]
        print(f"\nSample {i+1}:")
        print(f"  Filename: {sample['filename']}")
        print(f"  Categories ({len(cat_names)}): {', '.join(cat_names)}")

verify_samples(final_base_samples, cat_id_to_name, "Base Set")
verify_samples(final_val_samples, cat_id_to_name, "Validation Set")
verify_samples(final_novel_samples, cat_id_to_name, "Novel Set")

## Summary

### Output Files
- coco_mutually_exclusive_split.json: complete dataset split
- category_mapping.json: category-to-ID mapping
- split_report.txt: summary statistics
